In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv("Cust_Segmentation")
merged=pd.read_csv('../data/processed/Merged_dataset.csv')
df.head(4)

,customer_unique_id,recency_days,frequency,monetary,Monetary_transformed,cluster,segment
0,0000366f3b9a7992bf8c76cfdf3221e2,115,1,129.90,0.427544,0,High Potential
1,0000b849f77a49e4a4ce2b2a4ca5be3f,118,1,18.90,-1.721846,2,High Value
2,0000f46a3911fa3c0805444483337064,541,1,69.00,-0.240897,1,Low Value
3,0000f6ccb0745a6a4b88665a16c9f078,325,1,25.99,-1.345204,2,High Value


In [3]:
df['churned']=df['recency_days']>120 # if recency days is greater than 180 then customer is most likely to be churned

# Repeat purchase behavior

In [4]:
df['repeat_customer']=df['frequency']>1 # if customer had puchased more than once then he is a repeat customer

In [5]:
df.head(2)

,customer_unique_id,recency_days,frequency,monetary,Monetary_transformed,cluster,segment,churned,repeat_customer
0,0000366f3b9a7992bf8c76cfdf3221e2,115,1,129.9,0.427544,0,High Potential,False,False
1,0000b849f77a49e4a4ce2b2a4ca5be3f,118,1,18.9,-1.721846,2,High Value,False,False


In [6]:
purchase_counts = (
    merged
    .groupby("customer_unique_id")["order_id"]
    .nunique()
    .reset_index(name="total_orders")
)
df['total_orders']=purchase_counts['total_orders']
df.head(2)

,customer_unique_id,recency_days,frequency,monetary,Monetary_transformed,cluster,segment,churned,repeat_customer,total_orders
0,0000366f3b9a7992bf8c76cfdf3221e2,115,1,129.9,0.427544,0,High Potential,False,False,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,118,1,18.9,-1.721846,2,High Value,False,False,1


In [7]:
df['churned'].value_counts()


churned
True     69366
False    24049
Name: count, dtype: int64

# Factors affecting retention

In [8]:
cust_profile=(
    merged
    .merge(df, on='customer_unique_id', how='inner')
)

In [9]:
cust_profile.head(2)

,customer_id,customer_unique_id,customer_state,customer_city,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,...,seller_state,recency_days,frequency,monetary,Monetary_transformed,cluster,segment,churned,repeat_customer,total_orders
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,SP,franca,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57.000000,2017-05-25 10:35:35.000000000,...,SP,474,1,124.99,0.387942,1,Low Value,True,False,1
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,SP,sao bernardo do campo,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59.000000,2018-01-29 12:41:19.000000000,...,SC,233,1,289.00,1.217067,0,High Potential,True,False,1


# Customer Churn analysis

### 1. Effect of Freight_value

In [10]:
(
    cust_profile
    .groupby('churned')
    .agg(
        avg_freight_value=('freight_value','mean'),
        median_freight_value=('freight_value','median'),
    )
    .reset_index()
)
# Customer having freight value less churned most

,churned,avg_freight_value,median_freight_value
0,False,21.580514,18.23
1,True,19.690422,16.13


### 2. Effect of Delivery Time 

In [11]:
(
    cust_profile
    .groupby('churned')
    .agg(
        mean_delivery_time=('delivery_time','mean'),
        median_delivery_time=('delivery_time','median'),
    )
    .reset_index()
)
# Churned customers have more average delivery time

,churned,mean_delivery_time,median_delivery_time
0,False,2.120013,2.079442
1,True,2.454632,2.484907


### 3. Effect of Late Delivery

In [12]:
cust_profile["Late Delivery"] = np.where(
    cust_profile["order_delivered_customer_date"] > cust_profile["order_estimated_delivery_date"],
    "Late",
    "On Time"
)

(
    cust_profile[cust_profile["Late Delivery"] == "Late"]
    .groupby('churned')
    .size()
    .reset_index(name="Late_delivery_count")
)

,churned,Late_delivery_count
0,False,1727
1,True,6487


### 3.Effect of Seller 

In [13]:
(
    cust_profile
    .groupby('churned')
    .agg(
        mean_review_score=('review_score','mean'),
    )
)
# Customer interaction with slightly worst performing seller churns the most

,mean_review_score
churned,
False,4.226410
True,4.024338


### 4. Effect of Payments Installments

In [14]:
(
    cust_profile
    .groupby('churned')
    .agg(
        mean_installments=('payment_installments','mean'),
    )
)
# Churned customers having on an average 3 installments are most likely to churn

,mean_installments
churned,
False,2.839170
True,2.922826


## Build at-risk customer profiles

In [15]:
cust_profile["at_risk"] = (
    (df["churned"] == False) & #customer not churned
    (df["recency_days"] > 120) & #customer hvaing recency greater than 120 days
    (df["frequency"] == 1) # customer who pruchased once
)

In [16]:
cust_profile.to_csv("../data/processed/Retention&Churn.csv",index=False)